# Stock Image Scraper 📸

In [1]:
# STEP 1:- Importing the required libraries
import time
import requests
import pandas as pd
from tqdm import tqdm
import chromedriver_binary
from bs4 import BeautifulSoup
from selenium import webdriver
from openpyxl import Workbook
import re
import os
import urllib.request
from urllib.parse import urljoin
import shutil

In [2]:
# STEP :-2:- Setting up the Selenium WebDriver
driver = webdriver.Chrome()
driver.get("https://stock-pictures.netlify.app/")
time.sleep(5)  # Wait for the page to load

The chromedriver version (147.0.7727.57) detected in PATH at c:\Users\Nbinary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\chromedriver_binary\chromedriver.exe might not be compatible with the detected chrome version (148.0.7778.181); currently, chromedriver 148.0.7778.178 is recommended for chrome 148.*, so it is advised to delete the driver in PATH and retry


In [3]:
# STEP 3:- Scrapping the url
soup = BeautifulSoup(driver.page_source, "html.parser")
image_elements = soup.select('img.source-img[src$=".jpg"]')
print(f"Found {len(image_elements)} images on the page.")

Found 48 images on the page.


In [4]:
# STEP 3:- Scraping the image details
# image url, name tags, likes and comments on each image
image_data = []
for sp in soup.find_all('div', class_='container'):
    img = sp.find('img')
    
    # 1. Safely check if img exists AND has a src attribute before proceeding
    if img and img.get('src') and 'gif' not in img.get('src'):
        link = img.get('src')
        
        # 2. Safely extract tags (Default to empty string if missing)
        tags_div = sp.find('div', class_='tags')
        tags_text = ""
        if tags_div:
            # Assuming the first 7 chars were something like "Tags: "
            # A safer way is replacing the specific word, or just matching words
            raw_tags = tags_div.text[7:].strip().split(' ')
            tags_text = ' '.join(list(set(raw_tags)))
        
        # 3. Safely extract likes and comments (Default to 0 if missing)
        likes, comments = 0, 0
        likes_comments_div = sp.find('div', class_='likes-comments')
        
        if likes_comments_div:
            spans = likes_comments_div.find_all('span')
            
            # re.search(r'\d+', text) finds the first sequence of numbers in a string
            if len(spans) > 0:
                likes_match = re.search(r'\d+', spans[0].text)
                likes = int(likes_match.group()) if likes_match else 0
                
            if len(spans) > 1:
                comments_match = re.search(r'\d+', spans[1].text)
                comments = int(comments_match.group()) if comments_match else 0
        
        image_data.append([link, tags_text, likes, comments])

In [5]:
# STEP 4: Make in to a dataframe
df = pd.DataFrame(image_data, columns=['Image URL', 'Tags', 'Likes', 'Comments'])

In [6]:
df

,Image URL,Tags,Likes,Comments
0,https://cdn.pixabay.com/photo/2022/03/06/05/30...,"Clouds, Blue Sky, Atmosphere, Sky",196,55
1,https://cdn.pixabay.com/photo/2022/04/07/11/45...,"Ornithology, Hummingbird Bird,",76,20
2,https://cdn.pixabay.com/photo/2022/02/28/15/28...,"Subtropical Rainfall, Rainbow, Sea,",282,106
3,https://cdn.pixabay.com/photo/2022/04/04/02/52...,"Blossoms, Cherry Road, Sakura Japan,",42,11
4,https://cdn.pixabay.com/photo/2022/04/09/18/06...,"Plant Flower, Marguerite, Cape",39,15
5,https://cdn.pixabay.com/photo/2021/11/13/23/06...,Relaxing The Under Rest Tree,522,108
6,https://cdn.pixabay.com/photo/2022/04/07/02/56...,"Cottontail Wild Rabbit, Grass",51,10
7,https://cdn.pixabay.com/photo/2022/03/19/21/11...,"Plant Spring, Crocus, Flowers,",135,44
8,https://cdn.pixabay.com/photo/2022/04/09/17/30...,"Coffee, Drink, Café, Vacation, Table",26,5
9,https://cdn.pixabay.com/photo/2022/03/25/19/24...,"Waterfall, Fall, Light Nature, Epic,",96,21


In [7]:
# STEP 5: Close the Selenium WebDriver
driver.quit()

In [8]:
# STEP 6: Save the data to an Excel file
df.to_excel("stock_images_data.xlsx", index=False)

In [9]:
# STEP 7: Download the images to a local folder
# Create a folder to save the images
path=[]
os.makedirs("downloaded_images", exist_ok=True)
# Use a browser-like User-Agent to avoid being blocked by the server
download_headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}
# Download each image
for index, row in tqdm(df.iterrows(), total=len(df)):
    image_url = row['Image URL']
    if image_url.startswith('/'):
        image_url = urljoin("https://stock-pictures.netlify.app/", image_url)
    image_name = f"image_{index + 1}.jpg"
    save_path = os.path.join("downloaded_images", image_name)
    path.append(save_path)
    try:
        response = requests.get(image_url, headers=download_headers, timeout=30, stream=True)
        response.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=10240):
                if chunk:
                    f.write(chunk)
        print(f"Downloaded: {image_name}")
    except Exception as e:
        print(f"Failed to download {image_name}: {image_url} : {e}")

  2%|▏         | 1/48 [00:00<00:36,  1.29it/s]

Downloaded: image_1.jpg


  4%|▍         | 2/48 [00:01<00:37,  1.23it/s]

Downloaded: image_2.jpg


  6%|▋         | 3/48 [00:02<00:35,  1.26it/s]

...

Downloaded: image_46.jpg


 98%|█████████▊| 47/48 [00:29<00:00,  1.60it/s]

Downloaded: image_47.jpg


100%|██████████| 48/48 [00:30<00:00,  1.58it/s]

Downloaded: image_48.jpg


In [10]:
# STEP 8: Update the Excel file with the local image paths
df['Local Image Path'] = path
df.to_excel("stock_images_data.xlsx", index=False)

In [11]:
df

,Image URL,Tags,Likes,Comments,Local Image Path
0,https://cdn.pixabay.com/photo/2022/03/06/05/30...,"Clouds, Blue Sky, Atmosphere, Sky",196,55,downloaded_images\image_1.jpg
1,https://cdn.pixabay.com/photo/2022/04/07/11/45...,"Ornithology, Hummingbird Bird,",76,20,downloaded_images\image_2.jpg
2,https://cdn.pixabay.com/photo/2022/02/28/15/28...,"Subtropical Rainfall, Rainbow, Sea,",282,106,downloaded_images\image_3.jpg
3,https://cdn.pixabay.com/photo/2022/04/04/02/52...,"Blossoms, Cherry Road, Sakura Japan,",42,11,downloaded_images\image_4.jpg
4,https://cdn.pixabay.com/photo/2022/04/09/18/06...,"Plant Flower, Marguerite, Cape",39,15,downloaded_images\image_5.jpg
5,https://cdn.pixabay.com/photo/2021/11/13/23/06...,Relaxing The Under Rest Tree,522,108,downloaded_images\image_6.jpg
6,https://cdn.pixabay.com/photo/2022/04/07/02/56...,"Cottontail Wild Rabbit, Grass",51,10,downloaded_images\image_7.jpg
7,https://cdn.pixabay.com/photo/2022/03/19/21/11...,"Plant Spring, Crocus, Flowers,",135,44,downloaded_images\image_8.jpg
8,https://cdn.pixabay.com/photo/2022/04/09/17/30...,"Coffee, Drink, Café, Vacation, Table",26,5,downloaded_images\image_9.jpg
9,https://cdn.pixabay.com/photo/2022/03/25/19/24...,"Waterfall, Fall, Light Nature, Epic,",96,21,downloaded_images\image_10.jpg


In [12]:
# STEP 9: Clone the dataset
df_new = df.copy()
# two columns path and tags are there reset of the columns are removed
df_new = df_new[['Local Image Path', 'Tags']]
df_new.to_excel("stock_images_data_cloned.xlsx", index=False)
df_new

,Local Image Path,Tags
0,downloaded_images\image_1.jpg,"Clouds, Blue Sky, Atmosphere, Sky"
1,downloaded_images\image_2.jpg,"Ornithology, Hummingbird Bird,"
2,downloaded_images\image_3.jpg,"Subtropical Rainfall, Rainbow, Sea,"
3,downloaded_images\image_4.jpg,"Blossoms, Cherry Road, Sakura Japan,"
4,downloaded_images\image_5.jpg,"Plant Flower, Marguerite, Cape"
5,downloaded_images\image_6.jpg,Relaxing The Under Rest Tree
6,downloaded_images\image_7.jpg,"Cottontail Wild Rabbit, Grass"
7,downloaded_images\image_8.jpg,"Plant Spring, Crocus, Flowers,"
8,downloaded_images\image_9.jpg,"Coffee, Drink, Café, Vacation, Table"
9,downloaded_images\image_10.jpg,"Waterfall, Fall, Light Nature, Epic,"


In [17]:
# STEP 10: Make different folder for unique tags and move the images to the respective folders
paths=[]
for index, row in df_new.iterrows():
    image_path = row['Local Image Path']
    if not os.path.exists(image_path):
        print(f"Missing file, skipping: {image_path}")
        continue

    tags = [tag.strip().strip(',.') for tag in re.split(r',\s*', row['Tags']) if tag.strip()]
    for tag in tags:
        tag_folder = os.path.join("tagged_images", tag)
        paths.append(tag_folder)
        os.makedirs(tag_folder, exist_ok=True)
        shutil.copy2(image_path, tag_folder)

Missing file, skipping: downloaded_images\image_32.jpg


In [18]:
# STEP 11: Create a zip file of the tagged images folder
shutil.make_archive("tagged_images", 'zip', "tagged_images")
# length of the folder and files in tagedd images folder
total_images = 0
for root, dirs, files in os.walk("tagged_images"):
    total_images += len(files) 

print(f"Total images in tagged_images folder: {total_images}")

Total images in tagged_images folder: 363
